In [123]:
import pandas as pd
from datetime import datetime
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder


clients_info = pd.read_csv('clientes_info.csv')

In [124]:
clients_info['Fecha_Registro'] = pd.to_datetime(clients_info['Fecha_Registro'], dayfirst=True, errors='coerce')


clients_info

,ID_Cliente,Nombre,Apellido,Email,Fecha_Registro,Region,total_productos,total_gasto,gasto_promedio,Cancelada,Completa,Pendiente,forma_pago_1,forma_pago_2,forma_pago_3,forma_pago_4,forma_pago_5
0,1,Karisa,Cromett,kcromett0@imageshack.us,2023-11-19,Patagonia,41,287.77,26.160909,0,11,2,1,0,3,4,5
1,2,Lenette,Seabert,lseabert1@yahoo.co.jp,2023-05-07,Patagonia,9,103.28,34.426667,0,3,1,1,0,0,2,1
2,3,Buddy,Silverson,bsilverson2@howstuffworks.com,2023-03-27,Patagonia,23,160.24,22.891429,0,7,1,3,0,2,2,1
3,4,Dan,Parkin,dparkin3@virginia.edu,2023-10-26,Buenos Aires,30,230.16,32.880000,0,7,1,2,1,2,2,1
4,5,Conney,Cassella,ccassella4@who.int,2023-03-31,Centro,19,164.48,23.497143,0,7,2,1,1,3,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321,322,Hugh,Ortelt,hortelt8x@google.com.hk,2023-11-19,NEA,14,70.60,23.533333,0,3,1,0,0,2,2,0
322,323,Issiah,Schoales,ischoales8y@twitter.com,2023-11-21,Centro,28,231.98,25.775556,0,9,1,1,1,1,2,5
323,324,Nalani,Steggals,nsteggals8z@nhs.uk,2023-09-01,NEA,19,179.48,29.913333,0,6,1,2,1,1,2,1
324,325,Rowland,Riddington,rriddington90@friendfeed.com,2023-06-16,NEA,40,314.16,26.180000,0,12,0,2,1,3,4,2


In [125]:
date_format = "%Y-%m-%d"
last_date = clients_info['Fecha_Registro'].max()

clients_info['dias_en_sistema'] = last_date - clients_info['Fecha_Registro']

In [126]:
clients_info.head(5)

,ID_Cliente,Nombre,Apellido,Email,Fecha_Registro,Region,total_productos,total_gasto,gasto_promedio,Cancelada,Completa,Pendiente,forma_pago_1,forma_pago_2,forma_pago_3,forma_pago_4,forma_pago_5,dias_en_sistema
0,1,Karisa,Cromett,kcromett0@imageshack.us,2023-11-19,Patagonia,41,287.77,26.160909,0,11,2,1,0,3,4,5,40 days
1,2,Lenette,Seabert,lseabert1@yahoo.co.jp,2023-05-07,Patagonia,9,103.28,34.426667,0,3,1,1,0,0,2,1,236 days
2,3,Buddy,Silverson,bsilverson2@howstuffworks.com,2023-03-27,Patagonia,23,160.24,22.891429,0,7,1,3,0,2,2,1,277 days
3,4,Dan,Parkin,dparkin3@virginia.edu,2023-10-26,Buenos Aires,30,230.16,32.880000,0,7,1,2,1,2,2,1,64 days
4,5,Conney,Cassella,ccassella4@who.int,2023-03-31,Centro,19,164.48,23.497143,0,7,2,1,1,3,3,1,273 days


In [127]:
clients_info['dias_en_sistema'] = clients_info['dias_en_sistema'].dt.days

clients_info

,ID_Cliente,Nombre,Apellido,Email,Fecha_Registro,Region,total_productos,total_gasto,gasto_promedio,Cancelada,Completa,Pendiente,forma_pago_1,forma_pago_2,forma_pago_3,forma_pago_4,forma_pago_5,dias_en_sistema
0,1,Karisa,Cromett,kcromett0@imageshack.us,2023-11-19,Patagonia,41,287.77,26.160909,0,11,2,1,0,3,4,5,40
1,2,Lenette,Seabert,lseabert1@yahoo.co.jp,2023-05-07,Patagonia,9,103.28,34.426667,0,3,1,1,0,0,2,1,236
2,3,Buddy,Silverson,bsilverson2@howstuffworks.com,2023-03-27,Patagonia,23,160.24,22.891429,0,7,1,3,0,2,2,1,277
3,4,Dan,Parkin,dparkin3@virginia.edu,2023-10-26,Buenos Aires,30,230.16,32.880000,0,7,1,2,1,2,2,1,64
4,5,Conney,Cassella,ccassella4@who.int,2023-03-31,Centro,19,164.48,23.497143,0,7,2,1,1,3,3,1,273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321,322,Hugh,Ortelt,hortelt8x@google.com.hk,2023-11-19,NEA,14,70.60,23.533333,0,3,1,0,0,2,2,0,40
322,323,Issiah,Schoales,ischoales8y@twitter.com,2023-11-21,Centro,28,231.98,25.775556,0,9,1,1,1,1,2,5,38
323,324,Nalani,Steggals,nsteggals8z@nhs.uk,2023-09-01,NEA,19,179.48,29.913333,0,6,1,2,1,1,2,1,119
324,325,Rowland,Riddington,rriddington90@friendfeed.com,2023-06-16,NEA,40,314.16,26.180000,0,12,0,2,1,3,4,2,196


In [128]:
# Se va a intentar predecir el gasto total de un cliente
target = 'total_gasto'
posssible_pred_cols = ['Region', 'gasto_promedio','forma_pago_1', 'forma_pago_2', 'forma_pago_3', 'forma_pago_4', 'forma_pago_5', 'dias_en_sistema']


#aplciando LabelEncoder a la variable categorica (Region)
le = LabelEncoder()
clients_info['Region'] = le.fit_transform(clients_info['Region'])

In [129]:
#Model
y = clients_info[target]
X = clients_info[posssible_pred_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R2: {r2}")

Mean Squared Error: 3313.475431421515
R2: 0.7336444533129662
